In [1]:
import os
import time
import joblib
import numpy as np
import pandas as pd
from datetime import datetime

In [2]:
SELECTED_FEATURE_RANGES = {
    # ── Numerical features ──────────────────────────────────────────────────
    
    "sg":   {"type": "numeric", "min": 1.000,  "max": 1.035,
              "label": "Specific Gravity (sg)",
              "hint":  "e.g. 1.020  |  range: 1.000 – 1.035"},

    "al":   {"type": "numeric", "min": 0.0,    "max": 5.0,
              "label": "Albumin Level (al)",
              "hint":  "Scale 0 – 5  |  0 = absent, 5 = heavy"},

    "sc":   {"type": "numeric", "min": 0.1,    "max": 15.0,
              "label": "Serum Creatinine (sc) [mg/dL]",
              "hint":  "e.g. 1.2  |  normal: 0.6 – 1.2 mg/dL"},

    "pcv":  {"type": "numeric", "min": 15.0,   "max": 55.0,
              "label": "Packed Cell Volume (pcv) [%]",
              "hint":  "e.g. 44   |  normal: 35 – 50 %"},

    "rc":   {"type": "numeric", "min": 1.0,    "max": 8.0,
              "label": "Red Blood Cell Count (rc) [mill/mcL]",
              "hint":  "e.g. 5.1  |  normal: 4.5 – 5.5 mill/mcL"},
}

In [4]:
import os
import joblib
import numpy as np
import pandas as pd

# =========================================================================
# PHASE 1: AUTOMATIC PRODUCTION ASSETS LOADING
# =========================================================================
REQUIRED_FILES = {
    "model"   : "final_ckd_model.joblib",
    "scaler"  : "scaler.joblib",
    "features": "selected_features.joblib",
}

assets  = {}
missing = []

# Verify files exist locally
for asset_name, filename in REQUIRED_FILES.items():
    if os.path.exists(filename):
        assets[asset_name] = joblib.load(filename)
    else:
        missing.append(filename)

if missing:
    raise FileNotFoundError(
        f"Missing required production files: {missing}. "
        "Place your three .joblib files in the same folder as this notebook script."
    )

# Extract pipeline elements safely into cell memory
loaded_model   = assets["model"]       
loaded_scaler  = assets["scaler"]      
SELECTED_FEATS = assets["features"]    

if hasattr(loaded_scaler, "feature_names_in_"):
    SCALER_FEATS = list(loaded_scaler.feature_names_in_)
else:
    raise ValueError("Loaded scaler does not contain stored 'feature_names_in_' metadata.")


# =========================================================================
# PHASE 2: DEFINE EXPLICIT INFERENCE SYSTEM ENGINE
# =========================================================================
def run_portfolio_inference_pipeline(model_obj, scaler_obj, scaler_columns, model_features, feature_ranges):
    print("=========================================================================")
    print("             CHRONIC KIDNEY DISEASE (CKD) PATIENT DATA PORTAL            ")
    print("=========================================================================")
    print(" ENTER PATIENT METRICS BELOW AT THE SELECTION PROMPTS.                   ")
    print(" VALUES MUST REMAIN STRICTLY WITHIN THE PERMITTED CLINICAL BOUNDS.       ")
    print("=========================================================================\n")
    
    user_data = {}
    
    # SYSTEM INPUT LOOP - Collects metrics step-by-step using plain console output
    for feat, meta in feature_ranges.items():
        while True:
            if meta["type"] == "numeric":
                prompt_msg = f" -> Enter {meta['label']} [{meta['min']} to {meta['max']}]: "
            else:
                prompt_msg = f" -> Enter {meta['label']} [0 or 1]: "
            
            # --- LOCAL INTERACTIVE ENTRY / GITHUB STATIC FALLBACK ---
            try:
                raw_val = input(prompt_msg)
                if not raw_val: 
                    raise ValueError("Empty entry detected")
            except (NameError, ValueError, EOFError):
                # Fallback profile used to generate a clean, static report view on GitHub!
                fallback_defaults = {"sg": 1.020, "al": 0.0, "sc": 0.8, "pcv": 42.0, "rc": 4.8}
                raw_val = str(fallback_defaults[feat])
                print(f"{prompt_msg}{raw_val}") 
            
            # --- BOUNDARY RANGE VALIDATION ---
            try:
                val = float(raw_val)
                if meta["type"] == "numeric":
                    if not (meta["min"] <= val <= meta["max"]):
                        print(f"    ⚠️  ERROR: Input value ({val}) is out of bounds! Must be between {meta['min']} and {meta['max']}.\n")
                        continue
                elif meta["type"] == "binary":
                    if val not in [0.0, 1.0]:
                        print(f"    ⚠️  ERROR: Input value ({val}) must be exactly 0 or 1.\n")
                        continue
                
                user_data[feat] = val
                break 
            except ValueError:
                print("    ⚠️  ERROR: Invalid character entry. Please enter a valid number.\n")

    # Vector assembly and mathematical scaling
    full_row_dict = {col: 0.0 for col in scaler_columns}
    for feat, value in user_data.items():
        if feat in full_row_dict:
            full_row_dict[feat] = value
            
    inference_df = pd.DataFrame([full_row_dict], columns=scaler_columns)
    scaled_df = pd.DataFrame(scaler_obj.transform(inference_df), columns=scaler_columns)
    final_model_input = scaled_df[model_features]
    
    # Model Core Prediction Execution
    raw_pred = model_obj.predict(final_model_input)
    prediction = int(raw_pred[0] if hasattr(raw_pred, "__len__") else raw_pred)
    
    raw_probs = model_obj.predict_proba(final_model_input)
    confidence_score = float(raw_probs[0][prediction] if raw_probs.ndim > 1 else raw_probs[prediction])
    
    # Plain Text Report Construction
    border = "=" * 75
    sub_border = "-" * 75
    
    if prediction == 1:
        status_text = "HIGH RISK INDICATED (POSSIBLE CKD PRESENT)"
        guidance = (
            "  * Urgent Nephrology Referral: Recommend formal evaluation with a specialist.\n"
            "  * Confirmatory Testing: Schedule a repeat serum creatinine and Renal Ultrasound.\n"
            "  * Therapeutic Optimization: Focus strictly on BP titration. Avoid all NSAIDs.\n"
            "  * Dietary Modification: Initiate counseling regarding a renal-friendly diet framework."
        )
    else:
        status_text = "LOW RISK INDICATED (NO CKD PATTERN DETECTED)"
        guidance = (
            "  * Routine Monitoring: Maintain standard medical follow-up intervals. Assess eGFR annually.\n"
            "  * Preventative Management: Continue baseline tracking of metabolic biomarkers.\n"
            "  * Hydration and Wellness: Reinforce lifestyle choices supporting renal health preservation."
        )

    print("\n" + border)
    print("      CLINICAL DECISION SUPPORT SYSTEM (CDSS) DIAGNOSTIC REPORT      ")
    print(border)
    print(f" Patient Profile Metrics    : Processed successfully ({len(user_data)} attributes)")
    print(f" Diagnostic Classification  : {status_text}")
    print(f" Model Prediction Confidence: {confidence_score:.2%}")
    print(sub_border)
    print(" Evidence-Based Recommendations & Next Steps:")
    print(guidance)
    print(sub_border)
    print(" Disclaimer: This automated exploratory assessment is designed exclusively for")
    print(" screening and support. It does not replace independent clinical validation.")
    print(border)

# =========================================================================
# PHASE 3: EXECUTE PIPELINE
# =========================================================================
# Ensure SELECTED_FEATURE_RANGES is defined in your workspace before running this cell.
run_portfolio_inference_pipeline(
    model_obj=loaded_model, 
    scaler_obj=loaded_scaler, 
    scaler_columns=SCALER_FEATS, 
    model_features=SELECTED_FEATS, 
    feature_ranges=SELECTED_FEATURE_RANGES
)

             CHRONIC KIDNEY DISEASE (CKD) PATIENT DATA PORTAL            
 ENTER PATIENT METRICS BELOW AT THE SELECTION PROMPTS.                   
 VALUES MUST REMAIN STRICTLY WITHIN THE PERMITTED CLINICAL BOUNDS.       



 -> Enter Specific Gravity (sg) [1.0 to 1.035]:  1.02
 -> Enter Albumin Level (al) [0.0 to 5.0]:  0
 -> Enter Serum Creatinine (sc) [mg/dL] [0.1 to 15.0]:  4
 -> Enter Packed Cell Volume (pcv) [%] [15.0 to 55.0]:  45
 -> Enter Red Blood Cell Count (rc) [mill/mcL] [1.0 to 8.0]:  15


    ⚠️  ERROR: Input value (15.0) is out of bounds! Must be between 1.0 and 8.0.



 -> Enter Red Blood Cell Count (rc) [mill/mcL] [1.0 to 8.0]:  5



      CLINICAL DECISION SUPPORT SYSTEM (CDSS) DIAGNOSTIC REPORT      
 Patient Profile Metrics    : Processed successfully (5 attributes)
 Diagnostic Classification  : HIGH RISK INDICATED (POSSIBLE CKD PRESENT)
 Model Prediction Confidence: 64.67%
---------------------------------------------------------------------------
 Evidence-Based Recommendations & Next Steps:
  * Urgent Nephrology Referral: Recommend formal evaluation with a specialist.
  * Confirmatory Testing: Schedule a repeat serum creatinine and Renal Ultrasound.
  * Therapeutic Optimization: Focus strictly on BP titration. Avoid all NSAIDs.
  * Dietary Modification: Initiate counseling regarding a renal-friendly diet framework.
---------------------------------------------------------------------------
 Disclaimer: This automated exploratory assessment is designed exclusively for
 screening and support. It does not replace independent clinical validation.
